# PBMC 1k v3: FASTQ → AnnData

This notebook streams gzipped FASTQ files from `assets/pbmc_1k/pbmc_1k_v3_fastqs` and builds an **`AnnData`** object.

## Important limitation

A standard single-cell **`AnnData`** is **cells × genes** (counts per gene). Raw FASTQ only contains sequences and quality scores; **there is no gene matrix until you align/pseudoalign and count** (Cell Ranger, `STARsolo`, `alevin-fry`, `kallisto | bustools`, etc.).

Here we parse Chromium **3′ v3** read structure on **R1**: **16 bp cell barcode** + **12 bp UMI**, then aggregate **reads per cell barcode** into a one-feature matrix (`read_count`). Useful for QC, barcode saturation estimates, or as a template before you plug in a full counting pipeline.

Set `MAX_READS_PER_FILE` to `None` only if you intend to scan entire FASTQs (very slow, ~billions of reads).

In [ ]:
from __future__ import annotations

import gzip
from collections import Counter
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

In [ ]:
REPO_ROOT = Path("..").resolve()
FASTQ_DIR = REPO_ROOT / "assets" / "pbmc_1k" / "pbmc_1k_v3_fastqs"

# 10x Chromium 3' v3: R1 = 16 bp cell barcode + 12 bp UMI (+ optional technical sequence on longer reads)
CB_LEN = 16
UMI_LEN = 12

# Cap for interactive runs; use None to scan entire files (slow).
MAX_READS_PER_FILE: int | None = 100_000

OUT_H5AD = REPO_ROOT / "assets" / "pbmc_1k" / "pbmc_1k_v3_fastq_barcode_counts.h5ad"

list(sorted(FASTQ_DIR.glob("*.fastq.gz")))

In [ ]:
def iter_fastq_records(path: Path):
    """Yield (name, sequence, quality) for each read in a .fastq.gz file."""
    with gzip.open(path, "rt") as handle:
        while True:
            header = handle.readline()
            if not header:
                break
            seq = handle.readline()
            plus = handle.readline()
            qual = handle.readline()
            if not qual:
                break
            name = header.strip().removeprefix("@")
            yield name, seq.strip(), qual.strip()


def pair_r1_r2_paths(fastq_dir: Path) -> list[tuple[str, Path, Path]]:
    """Return (lane_id, R1, R2) for each lane; ignores index (I1) reads."""
    r1 = sorted(fastq_dir.glob("*_R1_*.fastq.gz"))
    out = []
    for p in r1:
        parts = p.name.split("_")
        # ..._S1_L001_R1_001.fastq.gz → find L*
        lane = next(x for x in parts if x.startswith("L") and x[1:].isdigit())
        r2_name = p.name.replace("_R1_", "_R2_")
        r2 = p.with_name(r2_name)
        if not r2.is_file():
            raise FileNotFoundError(f"Missing R2 for {p.name}: {r2}")
        out.append((lane, p, r2))
    return sorted(out, key=lambda t: t[0])


pair_r1_r2_paths(FASTQ_DIR)

In [ ]:
def count_reads_per_barcode(r1_path: Path, max_reads: int | None) -> Counter[str]:
    counts: Counter[str] = Counter()
    for i, (_name, seq, _qual) in enumerate(iter_fastq_records(r1_path)):
        if max_reads is not None and i >= max_reads:
            break
        if len(seq) < CB_LEN + UMI_LEN:
            continue
        cb = seq[:CB_LEN]
        counts[cb] += 1
    return counts


total_by_barcode: Counter[str] = Counter()
lane_stats: dict[str, dict] = {}

for lane, r1_path, r2_path in pair_r1_r2_paths(FASTQ_DIR):
    cb_counts = count_reads_per_barcode(r1_path, MAX_READS_PER_FILE)
    total_by_barcode.update(cb_counts)
    lane_stats[lane] = {
        "r1_path": str(r1_path.relative_to(REPO_ROOT)),
        "r2_path": str(r2_path.relative_to(REPO_ROOT)),
        "reads_used": sum(cb_counts.values()),
        "unique_barcodes": len(cb_counts),
    }

lane_stats

In [ ]:
if not total_by_barcode:
    raise ValueError("No barcodes counted — check FASTQ paths and MAX_READS_PER_FILE.")

barcodes = np.array(list(total_by_barcode.keys()))
counts = np.array([total_by_barcode[b] for b in barcodes], dtype=np.int64)

X = sp.csr_matrix(counts.reshape(-1, 1))
obs = pd.DataFrame(index=barcodes)
obs.index.name = "cell_barcode"
var = pd.DataFrame(index=["read_count"])

adata = ad.AnnData(X=X, obs=obs, var=var)
adata.uns["fastq_lane_stats"] = lane_stats
adata.uns["chemistry"] = "10x Chromium 3p v3 (R1: 16bp CB + 12bp UMI)"
adata.uns["note"] = (
    "X is one column: reads per barcode from R1 only. "
    "For genes × cells, run a aligner/counter and use sc.read_10x_* / h5."
)

adata

In [ ]:
adata.write_h5ad(OUT_H5AD)
OUT_H5AD